# Cooling Check: Weather-Normalized Building Anomaly Detection

## Framing

### Who it's for
**Facility operators** — people responsible for keeping buildings running efficiently and spotting when cooling systems aren't performing as expected.

### The Question
**Is this building's cooling running worse than the weather explains?**

More formally: given outdoor temperature and humidity, how much cooling energy *should* this building use? When actual usage exceeds that prediction, something's wrong (fouled heat exchanger, valve stuck, compressor inefficiency, etc.).

### What this is NOT
- Not production code or a real-time monitoring system
- Not a perfect physics model (it's simplified, linear)
- Not a replacement for domain expertise or maintenance schedules
- Not a control system or automated setpoint optimizer

---

## Why Weather-Normalize?

**The Problem**: Comparing raw cooling energy month-to-month is misleading.
- If it's hotter, you use more cooling. That's not a problem; that's physics.
- If it's more humid, cooling towers work harder. Again, expected.
- A "20% higher energy use" month might just be a hotter month.

**The Solution**: Build a model that explains cooling load as a function of weather. Then compare actual vs predicted.
- If actual > predicted by 10%, that's an anomaly worth investigating.
- If actual < predicted, maybe the system is over-designed or controls are tight.

**Why this matters for operators**:
- Focuses attention on real problems, not weather noise.
- Enables early detection of degradation before full failure.
- Helps justify maintenance budgets ("we found this before it broke").

---

## Approach

1. **Physics baseline**: Fit a change-point model to relate cooling load to outdoor temperature (ASHRAE Guideline 14).
2. **ML on the gap**: Model the residual (actual − baseline) to capture hour-of-day, day-of-week, humidity effects.
3. **Uncertainty quantification**: Provide prediction bands so operators know the confidence.
4. **Time-series validation**: Backtest on held-out months to ensure we're not overfitting.
5. **Cold-start capability**: Show how to onboard a new building with just 2–4 weeks of data.
6. **Drift detection**: Monitor for slow degradation or equipment changes.
7. **Actionable output**: Flag hours outside the band, convert excess to $ and CO₂.


---

## Section 2: Data

**Why one site?** ASHRAE GEPIII has 16 sites. Picking one keeps the scope tight for a portfolio piece, lets us understand local weather patterns well, and shows we can think about sample bias. (Full production would pool across sites, with site-level random effects.)

**Which site?** Site 0: 24 buildings with chilled water, Education + Office + Lodging mix, ~200K sq ft (large enough to show real patterns, not noise).

**Why these 7 buildings?** Largest by square footage (200K–130K sq ft), diverse use types, good data coverage.

### Raw Data Inspection

Before cleaning, let's look at what we have:

In [ ]:
Coming next: psychrometric calculations and plots

### Key Takeaways

**Answer out loud:**

1. **Why does humidity matter for cooling load?**
   - Cooling towers reject heat by evaporating water. Humid air (low dew point) can't accept much more moisture → evaporation slows → cooling capacity drops.
   - Enthalpy captures this: latent heat (moisture) is a huge part of the total heat content in humid air.
   - So a hot, dry day and a hot, humid day at the same temperature have *very* different cooling loads.

2. **Why does wet-bulb matter for cooling towers?**
   - Cooling tower performance is limited by the approach to wet-bulb temperature (the coldest the tower can cool water).
   - On a humid day (high wet-bulb), the tower can't cool water as far below the outdoor temperature.
   - So wet-bulb directly constrains how hard the chiller has to work.

**For the model**: weather-normalized cooling load should use the feature that best captures operational physics. Our data shows [wet-bulb or enthalpy?] tracks load better than dry-bulb alone. This will be our physics-based predictor for Section 4.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cooling load by hour of day
hourly_mean = data.groupby('hour')['meter_reading'].mean()
hourly_std = data.groupby('hour')['meter_reading'].std()
axes[0].plot(hourly_mean.index, hourly_mean.values, 'o-', linewidth=2, markersize=6, color='steelblue')
axes[0].fill_between(hourly_mean.index, 
                      hourly_mean.values - hourly_std.values,
                      hourly_mean.values + hourly_std.values,
                      alpha=0.2, color='steelblue')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Cooling Load (kWh)')
axes[0].set_title('Cooling Load Profile: Hour of Day')
axes[0].grid(True, alpha=0.3)

# Cooling load by day of week
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_mean = data.groupby('day_of_week')['meter_reading'].mean()
dow_std = data.groupby('day_of_week')['meter_reading'].std()
axes[1].bar(dow_mean.index, dow_mean.values, color=['steelblue']*5 + ['coral']*2, alpha=0.7)
axes[1].errorbar(dow_mean.index, dow_mean.values, yerr=dow_std.values, fmt='none', color='black', capsize=5)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_labels)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Mean Cooling Load (kWh)')
axes[1].set_title('Cooling Load Profile: Day of Week')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('operational_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cooling load by hour of day (mean ± std):")
for h in range(0, 24, 4):
    print(f"  {h:02d}:00 — {hourly_mean.iloc[h]:.0f} ± {hourly_std.iloc[h]:.0f}")

print("\nCooling load by day of week (mean ± std):")
for d in range(7):
    print(f"  {dow_labels[d]} — {dow_mean.iloc[d]:.0f} ± {dow_std.iloc[d]:.0f}")

### Physics Insights

Plot by hour and day of week to see occupancy + operational patterns:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter: dry bulb vs meter reading
axes[0].scatter(data['air_temperature'], data['meter_reading'], alpha=0.3, s=5)
axes[0].set_xlabel('Dry Bulb (°C)')
axes[0].set_ylabel('Cooling Load (kWh)')
axes[0].set_title('Load vs Dry Bulb')
axes[0].grid(True, alpha=0.3)
r_temp = data[['air_temperature', 'meter_reading']].corr().iloc[0, 1]
axes[0].text(0.05, 0.95, f'r = {r_temp:.3f}', transform=axes[0].transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Scatter: wet bulb vs meter reading
axes[1].scatter(data['wet_bulb'], data['meter_reading'], alpha=0.3, s=5, color='orange')
axes[1].set_xlabel('Wet Bulb (°C)')
axes[1].set_ylabel('Cooling Load (kWh)')
axes[1].set_title('Load vs Wet Bulb')
axes[1].grid(True, alpha=0.3)
r_wb = data[['wet_bulb', 'meter_reading']].corr().iloc[0, 1]
axes[1].text(0.05, 0.95, f'r = {r_wb:.3f}', transform=axes[1].transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Scatter: enthalpy vs meter reading
axes[2].scatter(data['enthalpy'], data['meter_reading'], alpha=0.3, s=5, color='green')
axes[2].set_xlabel('Enthalpy (kJ/kg)')
axes[2].set_ylabel('Cooling Load (kWh)')
axes[2].set_title('Load vs Enthalpy')
axes[2].grid(True, alpha=0.3)
r_h = data[['enthalpy', 'meter_reading']].corr().iloc[0, 1]
axes[2].text(0.05, 0.95, f'r = {r_h:.3f}', transform=axes[2].transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('physics_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCorrelation with cooling load:")
print(f"  Dry bulb temperature:  r = {r_temp:.3f}")
print(f"  Wet bulb temperature: r = {r_wb:.3f}")
print(f"  Enthalpy:             r = {r_h:.3f}")
print(f"\nWinner: {'Wet bulb' if r_wb > r_h else 'Enthalpy'} (r = {max(r_wb, r_h):.3f})")

### Which One Tracks Cooling Load Better?

Plot cooling load vs dry bulb, wet bulb, and enthalpy. Look for the tightest relationship.

In [ ]:
def wet_bulb_temperature(dry_bulb, dew_point):
    """
    Calculate wet-bulb temperature from dry bulb and dew point.
    Uses Magnus formula approximation (valid for -40°C to 60°C).
    
    Inputs:
      dry_bulb: air temperature (°C)
      dew_point: dew point temperature (°C)
    Output:
      wet_bulb: wet-bulb temperature (°C)
    """
    a = 17.27
    b = 237.7  # °C
    
    # Relative humidity from dew point and dry bulb
    alpha_dp = (a * dew_point) / (b + dew_point)
    alpha_db = (a * dry_bulb) / (b + dry_bulb)
    rh = 100 * np.exp(alpha_dp - alpha_db)
    rh = np.clip(rh, 0, 100)  # Clamp to [0, 100]
    
    # Wet-bulb temperature (Stull formula, good approximation)
    wb = dry_bulb * np.arctan(0.151977 * (rh + 8.313659) ** 0.5) + \
         np.arctan(dry_bulb + rh) - \
         np.arctan(rh - 1.676331) + \
         0.00391838 * rh ** 1.5 * np.arctan(0.023101 * rh) - 4.686035
    
    return wb

def enthalpy(dry_bulb, dew_point):
    """
    Calculate enthalpy of moist air (kJ/kg dry air).
    Uses ASHRAE equation: h = 1.006 * T + W * (2501 + 1.86 * T)
    where W is humidity ratio (kg water / kg dry air).
    
    Inputs:
      dry_bulb: air temperature (°C)
      dew_point: dew point temperature (°C)
    Output:
      h: enthalpy (kJ/kg dry air)
    """
    # Saturation pressure at dry bulb and dew point (Pa, Magnus formula)
    def vapor_pressure(T):
        return 610.5 * np.exp((17.27 * T) / (237.7 + T))
    
    P_w_sat = vapor_pressure(dew_point)  # Saturation pressure at dew point
    P_atm = 101325  # Pa (standard atmospheric pressure)
    
    # Humidity ratio (kg water / kg dry air)
    W = 0.622 * P_w_sat / (P_atm - P_w_sat)
    
    # Enthalpy (kJ/kg dry air)
    h = 1.006 * dry_bulb + W * (2501 + 1.86 * dry_bulb)
    
    return h

# Calculate features
data['wet_bulb'] = wet_bulb_temperature(data['air_temperature'], data['dew_temperature'])
data['enthalpy'] = enthalpy(data['air_temperature'], data['dew_temperature'])

print("Psychrometric features calculated:")
print(f"Wet-bulb range: {data['wet_bulb'].min():.1f}°C to {data['wet_bulb'].max():.1f}°C")
print(f"Enthalpy range: {data['enthalpy'].min():.1f} to {data['enthalpy'].max():.1f} kJ/kg")
print(f"\nSample rows:")
print(data[['timestamp', 'meter_reading', 'air_temperature', 'dew_temperature', 'wet_bulb', 'enthalpy']].head(10))

### Psychrometric Calculations

Why humidity matters: cooling towers reject heat to the air. If the air is already humid, the tower can't evaporate water as efficiently → cooling load rises. We need a metric that captures both temperature and humidity.

**Wet-bulb temperature**: the temperature you'd measure if you wrapped a thermometer in a wet cloth (equilibrium of evaporative cooling). Higher wet-bulb = harder for cooling towers to work.

**Enthalpy**: total heat content of the air (sensible + latent). Higher enthalpy = more energy to cool the air to a given setpoint.

Let's calculate both:

---

## Section 3: Physics Features

### Data Cleaning Decisions: Summary

| Decision | Reason | Consequence |
|----------|--------|-------------|
| Keep only Site 0 | Manageable scope, understand local weather | Lose generalization; production would pool sites |
| Select 7 buildings (largest by sq ft) | Good signal (large buildings), diverse mix | Selection bias toward modern, big buildings |
| Drop leading zeros | Equipment startup or missing data, not real cooling | Lose Jan–early Mar data, bias toward summer |
| Keep only air_temp & dew_point | Other weather features >50% missing | Lose cloud cover, wind effects (acceptable trade) |
| Merge on inner join (weather ∩ energy) | Both must have data | Lose a few hours with missing weather |
| **No imputation** | Cleaning decision #1: be honest about gaps | Smaller dataset, but no synthetic data |

**Answer out loud:**
1. **Why one site and not all?** Manageable scope for a portfolio piece. Shows I understand sample bias. Production would pool sites with hierarchical effects.
2. **What did you drop, and could that bias results?** Leading zeros (Jan–Feb data → summer bias), buildings <200K sq ft (large-building bias), features with >50% missing (can't use cloud/wind). Summer bias is real—we'll validate on months with full coverage in the backtest (Section 7).

In [ ]:
# Combine all buildings' data with weather
all_data = []
for b_id in SELECTED_BUILDINGS:
    df = cleaned_data[b_id][['timestamp', 'meter_reading']].copy()
    df['building_id'] = b_id
    all_data.append(df)

energy_df = pd.concat(all_data, ignore_index=True).sort_values(['building_id', 'timestamp'])

# Merge with weather
data = energy_df.merge(weather_use, on='timestamp', how='inner')

# Add time features for later
data['hour'] = data['timestamp'].dt.hour
data['day_of_week'] = data['timestamp'].dt.dayofweek
data['month'] = data['timestamp'].dt.month
data['is_weekend'] = data['day_of_week'] >= 5

# Get building metadata
data = data.merge(metadata[['building_id', 'primary_use', 'square_feet']],
                  on='building_id', how='left')

print(f"Final dataset shape: {data.shape}")
print(f"Date range: {data['timestamp'].min()} to {data['timestamp'].max()}")
print(f"Buildings: {data['building_id'].nunique()}")
print(f"Hours per building: {data.groupby('building_id').size()}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nFirst rows:")
print(data.head())

### Final Dataset: Merge and Align

In [ ]:
# Weather for Site 0
weather = weather_raw[weather_raw['site_id'] == SITE_ID].copy()
weather['timestamp'] = pd.to_datetime(weather['timestamp'])

# Keep columns needed for physics model
# air_temperature: outdoor dry bulb (input to change-point model)
# dew_temperature: for humidity calculations (wet-bulb, enthalpy)
# Others have too many missing values (cloud_coverage 78% missing)
weather_use = weather[['timestamp', 'air_temperature', 'dew_temperature']].copy()

# Drop rows with missing temperature (we need this for the model)
initial_weather = len(weather_use)
weather_use = weather_use.dropna(subset=['air_temperature', 'dew_temperature'])
dropped_weather = initial_weather - len(weather_use)

print(f"Weather: {len(weather_use)} hours (dropped {dropped_weather} with missing temp)")
print(f"Date range: {weather_use['timestamp'].min()} to {weather_use['timestamp'].max()}")

# Sanity check: temperature range
print(f"\nTemperature range: {weather_use['air_temperature'].min():.1f}°C to {weather_use['air_temperature'].max():.1f}°C")
print(f"Dew point range: {weather_use['dew_temperature'].min():.1f}°C to {weather_use['dew_temperature'].max():.1f}°C")

### Weather Data

Pull weather for Site 0, keeping only the columns we'll use:

In [ ]:
def clean_building_data(building_id, meter_1_df):
    """
    Clean chilled water data for one building.
    Returns: cleaned DataFrame, metadata about the cleaning
    """
    df = meter_1_df[meter_1_df['building_id'] == building_id].copy().sort_values('timestamp')
    
    initial_count = len(df)
    
    # Drop leading zeros (equipment startup or missing data)
    df = df[df['meter_reading'] > 0].reset_index(drop=True)
    leading_zero_count = initial_count - len(df)
    
    # Parse timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Drop duplicates (same timestamp and building = error in raw data)
    df = df.drop_duplicates(subset=['building_id', 'timestamp'], keep='first')
    dup_count = initial_count - len(df) - leading_zero_count
    
    # Check for gaps > 1 hour (we expect hourly data)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['hour_gap'] = df['timestamp'].diff().dt.total_seconds() / 3600
    gap_hours = df[df['hour_gap'] > 1.5]
    gap_count = len(gap_hours)
    
    if gap_count > 0:
        print(f"  Building {building_id}: {gap_count} gaps > 1.5 hours")
    
    return df, {
        'initial': initial_count,
        'leading_zeros': leading_zero_count,
        'duplicates': dup_count,
        'gaps': gap_count,
        'final': len(df)
    }

# Clean all selected buildings
cleaned_data = {}
cleaning_log = []

for b_id in SELECTED_BUILDINGS:
    df, log = clean_building_data(b_id, meter_1)
    cleaned_data[b_id] = df
    log['building_id'] = b_id
    cleaning_log.append(log)
    
cleaning_summary = pd.DataFrame(cleaning_log)
print("Cleaning summary:")
print(cleaning_summary[['building_id', 'initial', 'leading_zeros', 'duplicates', 'gaps', 'final']])

### Data Cleaning Decisions

**Decision 1: Handle leading zeros**
- Building 95 has 393 consecutive zeros (Feb 29 – Mar 14)
- **Why**: This looks like equipment startup or missing data, not real zero cooling
- **Action**: Drop all records before first non-zero reading for each building
- **Could this bias results?** Yes—we're biasing toward summer (peak cooling). Mitigation: we'll backtest on months with full data (Section 7).

In [ ]:
# Pick one building and look at raw rows
b_id = SELECTED_BUILDINGS[0]  # Building 95
raw = meter_1[meter_1['building_id'] == b_id].sort_values('timestamp').reset_index(drop=True)

print(f"Building {b_id}: {len(raw)} hours of data")
print(f"Date range: {raw['timestamp'].min()} to {raw['timestamp'].max()}")
print(f"\nFirst 10 hours:")
print(raw.head(10))

print(f"\n--- Check 1: Zero readings ---")
zero_ct = (raw['meter_reading'] == 0).sum()
print(f"Zero readings: {zero_ct} / {len(raw)} ({100*zero_ct/len(raw):.1f}%)")
print("First rows with zeros:")
zero_rows = raw[raw['meter_reading'] == 0].head(20)
print(zero_rows[['timestamp', 'meter_reading']])

print(f"\n--- Check 2: Time gaps ---")
raw['ts'] = pd.to_datetime(raw['timestamp'])
raw['gap'] = raw['ts'].diff().dt.total_seconds() / 3600  # hours
gaps = raw[raw['gap'] > 1.5]
print(f"Records with gap > 1.5 hours: {len(gaps)}")
if len(gaps) > 0:
    print("First 10 gaps:")
    print(gaps[['timestamp', 'gap']].head(10))

print(f"\n--- Check 3: Value distribution ---")
print(raw['meter_reading'].describe())